# DDDNet denoising on real Tianmouc data

This example uses the current DDDNet student network, downloads the supplied checkpoint automatically, saves normalized denoised `[TD, SDX, SDY]` tensors, renders the requested 3-row × 2-column comparison, and writes an MP4 demo. Positive signal is green and negative signal is blue.

In [ ]:
from pathlib import Path
import os
import cv2
import numpy as np
import torch
from IPython.display import Image, display
from tqdm.auto import tqdm
from tianmoucv.data import TianmoucDataReader

from tianmoucv.proc.denoise import (
    TianmoucDenoiser, make_tsd_comparison, sample_to_tsd
)
from tianmoucv.utils import save_video

In [ ]:
# Point this at one real capture folder whose direct children are cone/ and rod/.
data_path = Path(os.environ.get(
    'TIANMOUC_DATA_PATH',
    '/data51/yihan/tianmoucData/tianmoucReconDataset/test/normal/outdoor_cross_6',
))
output_dir = Path('denoise_demo_output')
checkpoint = None  # None: download/cache DEFAULT_DENOISE_URL; a local path also works.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sample_start = 0
sample_count = 4  # Set None to process the complete sequence.
video_fps = 10
visualization_gain = 8.0

if not (data_path / 'cone').is_dir() or not (data_path / 'rod').is_dir():
    raise FileNotFoundError(f'Expected cone/ and rod/ under: {data_path}')
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'denoised_tsd').mkdir(exist_ok=True)
print('data:', data_path.resolve())
print('output:', output_dir.resolve())
print('device:', device)

In [ ]:
# New TianMouCV readers support these correction flags; retain compatibility with older releases.
try:
    dataset = TianmoucDataReader(
        str(data_path), N=1, strict=False, _correct_sd_upsampler=True
    )
except TypeError:
    dataset = TianmoucDataReader(str(data_path), N=1)

denoiser = TianmoucDenoiser(checkpoint=checkpoint, device=device)
print('samples:', len(dataset))
print('weights:', denoiser.checkpoint_path)

In [ ]:
stop = len(dataset) if sample_count is None else min(len(dataset), sample_start + sample_count)
comparison_frames = []
denoiser.reset_state()

for sample_index in tqdm(range(sample_start, stop), desc='DDDNet'):
    sample = dataset[sample_index]
    raw_tsd = sample_to_tsd(sample)  # [3,T,H,W], normalized to the model's scale
    result = denoiser(raw_tsd)
    denoised_tsd = result['denoised_tsd'][0].cpu()

    # One independently readable file per reader sample avoids keeping a full capture in RAM.
    np.savez_compressed(
        output_dir / 'denoised_tsd' / f'{sample_index:06d}.npz',
        denoised_tsd=denoised_tsd.numpy().astype(np.float32),
        source_index=np.int64(sample_index),
        normalization_scale=np.float32(128.0),
    )

    for t in range(raw_tsd.shape[1]):
        comparison_frames.append(
            make_tsd_comparison(
                raw_tsd, denoised_tsd, t, gain=visualization_gain,
                display_frame_index=len(comparison_frames),
            )
        )

video_path = save_video(comparison_frames, output_dir / 'denoise_demo.mp4', fps=video_fps)
preview_path = output_dir / 'comparison_preview.png'
cv2.imwrite(str(preview_path), cv2.cvtColor(comparison_frames[-1], cv2.COLOR_RGB2BGR))
print(f'Saved {stop - sample_start} denoised TSD sample(s) to {output_dir / "denoised_tsd"}')
print('Saved video:', video_path)
display(Image(filename=str(preview_path)))